# Context-Aware Chatbot Using LangChain and RAG

**Runs entirely on Google Colab Free — no paid APIs, no GPU required.**

This notebook builds the full project step by step:
1. Install libraries
2. Import libraries
3. Create sample documents
4. Generate embeddings
5. Build FAISS vector store
6. Test the chatbot (with conversational memory)
7. Save the vector database
8. Download all project files


## 1. Install libraries


In [ ]:
!pip install -q langchain==0.3.7 langchain-community==0.3.5 langchain-huggingface==0.1.2 \
    faiss-cpu==1.9.0 sentence-transformers==3.2.1 transformers==4.46.2 torch==2.5.1 \
    streamlit==1.39.0 pypdf==5.1.0 accelerate==1.1.1
print('Libraries installed successfully.')


## 2. Import libraries


In [ ]:
import os

from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

print('All libraries imported successfully.')


## 3. Create sample documents

No dataset download is needed. This cell creates the full project folder structure (`ContextAwareChatbot/`) and writes 4 ready-made sample documents into `documents/`, plus the complete `ingest.py`, `chatbot.py`, `app.py`, `requirements.txt`, and `README.md` files, so the whole project exists locally in this Colab session and can be downloaded at the end.

> If you'd rather use your own documents, upload `.txt` or `.pdf` files into the `ContextAwareChatbot/documents/` folder using the Colab file browser (left sidebar) before running the ingestion cell below.


In [ ]:
PROJECT_DIR = 'ContextAwareChatbot'
DOCS_DIR = os.path.join(PROJECT_DIR, 'documents')
os.makedirs(DOCS_DIR, exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, 'vectorstore'), exist_ok=True)
print(f'Created project folders at ./{PROJECT_DIR}/')


In [ ]:
%%writefile ContextAwareChatbot/documents/ai.txt
Artificial Intelligence (AI)

Artificial Intelligence, or AI, is a branch of computer science that focuses on building machines and software capable of performing tasks that normally require human intelligence. These tasks include understanding language, recognizing images, making decisions, and solving problems.

History of AI
The term "Artificial Intelligence" was coined in 1956 at a conference held at Dartmouth College. Since then, AI has gone through several waves of enthusiasm and disappointment, often called "AI winters," followed by periods of rapid progress driven by better algorithms, more data, and faster hardware.

Types of AI
AI can be broadly divided into three categories.
Narrow AI, also called weak AI, is designed to perform a specific task, such as facial recognition or playing chess. Almost all AI systems in use today fall into this category.
General AI, also called strong AI, refers to a machine that could perform any intellectual task a human can do. This form of AI does not yet exist.
Super AI is a hypothetical form of AI that would surpass human intelligence in every field. It remains a topic of theoretical discussion.

Applications of AI
AI is used in many industries today. In healthcare, AI helps doctors diagnose diseases from medical images. In finance, AI detects fraudulent transactions. In transportation, AI powers self driving cars. In customer service, AI chatbots answer customer questions automatically. In agriculture, AI helps predict crop yields and detect plant diseases early.

Key AI Techniques
Machine learning allows systems to learn patterns from data without being explicitly programmed. Deep learning uses artificial neural networks with many layers to learn complex patterns. Natural language processing enables machines to understand and generate human language. Computer vision allows machines to interpret and understand visual information from the world.

Challenges in AI
Despite its progress, AI faces several challenges. Bias in training data can lead to unfair outcomes. Lack of transparency in complex models makes it hard to explain decisions. Data privacy concerns arise when AI systems process personal information. Ensuring AI safety and alignment with human values is an ongoing research area.

The Future of AI
Researchers continue to work on making AI systems more efficient, explainable, and fair. Many experts believe AI will continue to transform industries such as healthcare, education, and manufacturing over the coming decades.


In [ ]:
%%writefile ContextAwareChatbot/documents/ml.txt
Machine Learning (ML)

Machine Learning is a subset of Artificial Intelligence that gives computers the ability to learn from data and improve their performance over time without being explicitly programmed for every task.

How Machine Learning Works
A machine learning system is trained using a dataset. During training, the algorithm looks for patterns and relationships in the data. Once trained, the model can make predictions or decisions when given new, unseen data.

Types of Machine Learning
Supervised learning uses labeled data, where each training example has an input and a known correct output. The model learns to map inputs to outputs. Common supervised learning tasks include classification, such as spam detection, and regression, such as predicting house prices.
Unsupervised learning uses data without labels. The algorithm tries to find hidden patterns or groupings in the data on its own. Clustering and dimensionality reduction are common unsupervised learning techniques.
Reinforcement learning involves an agent that learns by interacting with an environment. The agent receives rewards or penalties based on its actions and learns to maximize the total reward over time. Reinforcement learning is used in robotics and game playing.

Common Machine Learning Algorithms
Linear regression predicts a continuous value based on input features. Logistic regression is used for binary classification problems. Decision trees split data into branches to make predictions. Random forests combine many decision trees to improve accuracy. Support vector machines find the best boundary to separate classes of data. K nearest neighbors classifies new data points based on the closest training examples.

The Machine Learning Workflow
A typical machine learning project involves several steps. First, data is collected and cleaned. Second, features are selected or engineered. Third, the data is split into training and testing sets. Fourth, a model is trained on the training set. Fifth, the model is evaluated on the testing set using metrics such as accuracy, precision, and recall. Finally, the model is deployed for real world use.

Overfitting and Underfitting
Overfitting occurs when a model learns the training data too well, including noise, and performs poorly on new data. Underfitting occurs when a model is too simple to capture the underlying patterns in the data. Techniques such as cross validation, regularization, and pruning help address these issues.

Applications of Machine Learning
Machine learning powers recommendation systems on streaming platforms, spam filters in email, credit scoring in banking, predictive maintenance in manufacturing, and personalized marketing in e commerce.


In [ ]:
%%writefile ContextAwareChatbot/documents/python.txt
Python Programming Language

Python is a high level, general purpose programming language created by Guido van Rossum and first released in 1991. It is known for its clean, readable syntax, which makes it a popular choice for beginners and experienced developers alike.

Key Features of Python
Python has a simple and readable syntax that closely resembles plain English. It is an interpreted language, meaning code is executed line by line without a separate compilation step. Python is dynamically typed, so variable types are determined automatically at runtime. It supports multiple programming paradigms, including procedural, object oriented, and functional programming. Python has a huge standard library and an extensive ecosystem of third party packages available through the Python Package Index, known as PyPI.

Why Python is Popular in AI and Data Science
Python has become the leading language for artificial intelligence, machine learning, and data science. Libraries such as NumPy and Pandas make numerical computing and data manipulation easy. Libraries such as scikit learn, TensorFlow, and PyTorch provide powerful tools for building machine learning and deep learning models. Frameworks such as LangChain make it easy to build applications powered by large language models. Jupyter Notebooks and Google Colab provide interactive environments for writing and testing Python code.

Basic Python Syntax
Variables in Python do not need explicit type declarations. Indentation is used to define code blocks instead of curly braces. Functions are defined using the def keyword. Classes are defined using the class keyword for object oriented programming. Python supports list comprehensions, which allow concise creation of lists.

Common Python Data Structures
Lists are ordered, mutable collections of items. Tuples are ordered, immutable collections of items. Dictionaries store data as key value pairs. Sets are unordered collections of unique items.

Python in Web and Software Development
Python is also widely used outside of AI. Frameworks such as Django and Flask are used to build web applications. Python is used for automation scripts, task scheduling, and testing. It is also used in scientific computing, game development, and embedded systems through tools like MicroPython.

Python Package Management
Python packages can be installed using pip, the standard package manager. Virtual environments, created using tools such as venv or conda, allow developers to manage dependencies separately for different projects, avoiding conflicts between package versions.


In [ ]:
%%writefile ContextAwareChatbot/documents/langchain.txt
LangChain Framework

LangChain is an open source framework designed to help developers build applications powered by large language models. It provides tools to connect language models with external data sources, memory, and other components, making it easier to build complex, context aware applications such as chatbots and question answering systems.

Why LangChain is Useful
Large language models on their own only know what they learned during training and have no memory of past conversations by default. LangChain solves this by providing building blocks such as document loaders, text splitters, vector stores, memory modules, and chains that can be combined to create powerful applications that go beyond simple text generation.

Core Components of LangChain
Document loaders read data from many different sources, such as text files, PDFs, websites, and databases, and convert them into a standard format that LangChain can process. Text splitters break large documents into smaller chunks so that they can be embedded and searched efficiently, since language models have a limited context window. Embeddings convert text into numerical vectors that capture semantic meaning, allowing similar pieces of text to be found using mathematical similarity. Vector stores, such as FAISS or Chroma, store these embeddings and allow fast similarity search to retrieve relevant chunks of text. Chains combine multiple steps, such as retrieving documents and generating an answer, into a single pipeline. Memory modules allow an application to remember previous interactions in a conversation, enabling more natural, context aware dialogue. Agents allow a language model to decide which tools or actions to take based on the user's request.

Retrieval Augmented Generation with LangChain
Retrieval Augmented Generation, or RAG, is a technique where relevant information is retrieved from an external knowledge base and provided to the language model as additional context before it generates a response. LangChain makes it straightforward to build RAG pipelines by combining document loaders, text splitters, embeddings, vector stores, and language models into a single retrieval chain. This approach allows a chatbot to answer questions using information from documents it was never directly trained on, and it reduces the chances of the model generating incorrect or made up information.

Conversational Memory in LangChain
LangChain provides several types of memory, including ConversationBufferMemory, which stores the entire conversation history, and more advanced memory types that summarize or selectively store information. This allows a chatbot to understand references to earlier parts of a conversation, such as pronouns referring to previously mentioned topics.

LangChain Ecosystem
LangChain integrates with many language model providers, including open source models hosted through HuggingFace, as well as many vector databases, document sources, and deployment tools such as Streamlit, making it a flexible framework for building real world AI applications.


### Write `requirements.txt`, `ingest.py`, `chatbot.py`, `app.py`, and `README.md` to disk


In [ ]:
%%writefile ContextAwareChatbot/requirements.txt
langchain==0.3.7
langchain-community==0.3.5
langchain-huggingface==0.1.2
faiss-cpu==1.9.0
sentence-transformers==3.2.1
transformers==4.46.2
torch==2.5.1
streamlit==1.39.0
pypdf==5.1.0
accelerate==1.1.1


In [ ]:
%%writefile ContextAwareChatbot/ingest.py
"""
ingest.py
---------
This script is responsible for the "ingestion" phase of the RAG pipeline.

What it does, step by step:
1. Loads all documents (.txt and .pdf) from the documents/ folder.
2. Splits each document into small overlapping chunks (so the retriever
   can find precise, relevant pieces of text instead of whole documents).
3. Converts each chunk into a numerical vector (embedding) using a free
   HuggingFace sentence-transformers model.
4. Stores all the vectors inside a FAISS vector database (fully local,
   no paid service required).
5. Saves the FAISS index to disk (vectorstore/) so chatbot.py can load
   it later without repeating this work.

Run this script once before running the chatbot:
    python ingest.py
"""

import os
import sys

# LangChain document loaders - read raw files into LangChain "Document" objects
from langchain_community.document_loaders import TextLoader, PyPDFLoader, DirectoryLoader

# Splits long documents into smaller overlapping chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Free, local sentence-transformers embedding model wrapper
from langchain_huggingface import HuggingFaceEmbeddings

# FAISS vector store wrapper (local, free, no server required)
from langchain_community.vectorstores import FAISS

# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------
DOCUMENTS_DIR = "documents"          # folder containing source documents
VECTORSTORE_DIR = "vectorstore"      # folder where the FAISS index will be saved
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

CHUNK_SIZE = 500        # number of characters per chunk
CHUNK_OVERLAP = 50      # overlap between chunks so context isn't lost at the edges


def load_documents(documents_dir: str):
    """
    Loads every .txt and .pdf file inside `documents_dir` and returns
    a list of LangChain Document objects.

    Error handling:
    - If the folder does not exist, or contains no supported files,
      a clear error is raised instead of failing silently.
    """
    if not os.path.isdir(documents_dir):
        raise FileNotFoundError(
            f"The documents folder '{documents_dir}' was not found. "
            f"Please create it and add at least one .txt or .pdf file."
        )

    all_docs = []

    # Load all .txt files
    try:
        txt_loader = DirectoryLoader(
            documents_dir,
            glob="**/*.txt",
            loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"},
            show_progress=False,
        )
        all_docs.extend(txt_loader.load())
    except Exception as e:
        print(f"[WARNING] Could not load .txt files: {e}")

    # Load all .pdf files
    try:
        pdf_files = [
            f for f in os.listdir(documents_dir) if f.lower().endswith(".pdf")
        ]
        for pdf_file in pdf_files:
            pdf_path = os.path.join(documents_dir, pdf_file)
            pdf_loader = PyPDFLoader(pdf_path)
            all_docs.extend(pdf_loader.load())
    except Exception as e:
        print(f"[WARNING] Could not load .pdf files: {e}")

    if len(all_docs) == 0:
        raise ValueError(
            f"No documents were found in '{documents_dir}'. "
            f"Add at least one .txt or .pdf file and try again."
        )

    print(f"[INFO] Loaded {len(all_docs)} document(s) from '{documents_dir}'.")
    return all_docs


def split_documents(documents):
    """
    Splits documents into smaller overlapping chunks.

    Why we do this:
    Language models and retrievers work better with short, focused pieces
    of text rather than very long documents. Overlap ensures that context
    at chunk boundaries is not lost.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(documents)
    print(f"[INFO] Split documents into {len(chunks)} chunks.")
    return chunks


def build_vectorstore(chunks):
    """
    Converts text chunks into embeddings and stores them in a FAISS index.
    """
    print(f"[INFO] Loading embedding model '{EMBEDDING_MODEL_NAME}' (this may take a moment)...")
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

    print("[INFO] Creating FAISS vector store from chunks...")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    return vectorstore


def save_vectorstore(vectorstore, path: str):
    """
    Saves the FAISS index to disk so it can be reloaded later without
    re-computing embeddings every time.
    """
    os.makedirs(path, exist_ok=True)
    vectorstore.save_local(path)
    print(f"[INFO] Vector store saved to '{path}'.")


def main():
    """
    Runs the full ingestion pipeline: load -> split -> embed -> save.
    """
    try:
        documents = load_documents(DOCUMENTS_DIR)
        chunks = split_documents(documents)
        vectorstore = build_vectorstore(chunks)
        save_vectorstore(vectorstore, VECTORSTORE_DIR)
        print("[SUCCESS] Ingestion complete! You can now run chatbot.py or app.py.")
    except Exception as e:
        print(f"[ERROR] Ingestion failed: {e}")
        sys.exit(1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ContextAwareChatbot/chatbot.py
"""
chatbot.py
----------
This module contains the core chatbot logic:

1. Loads the FAISS vector store created by ingest.py.
2. Loads a free, local HuggingFace LLM (google/flan-t5-base by default).
3. Builds a Conversational Retrieval Chain that combines:
     - the retriever (finds relevant chunks from the documents)
     - the LLM (generates the final answer)
     - conversation memory (remembers previous turns, so the bot
       understands follow-up questions like "where was he born?")
4. Exposes a simple ChatBot class that app.py (Streamlit) or a notebook
   can use with a single `chatbot.ask("your question")` call.

This file is written so it can be imported directly, or run on its own
for a quick command-line test:
    python chatbot.py
"""

import os

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------
VECTORSTORE_DIR = "vectorstore"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# flan-t5-base is small, free, and runs comfortably on Google Colab's free CPU/GPU.
# You can switch to "google/flan-t5-large" if you have a GPU runtime for better quality.
LLM_MODEL_NAME = "google/flan-t5-base"

# Number of relevant chunks to retrieve from the vector store per question
TOP_K_CHUNKS = 3

# A custom prompt that tells the model to answer using retrieved context only,
# and to say so honestly if the answer isn't in the documents.
QA_PROMPT_TEMPLATE = """You are a helpful assistant that answers questions using the
provided context from the user's documents. If the answer is not contained in the
context, say "I could not find that information in the provided documents." Do not
make up information.

Context:
{context}

Chat History:
{chat_history}

Question: {question}
Helpful Answer:"""


class ChatBot:
    """
    A context-aware chatbot that combines retrieval-augmented generation (RAG)
    with conversational memory.
    """

    def __init__(
        self,
        vectorstore_dir: str = VECTORSTORE_DIR,
        embedding_model_name: str = EMBEDDING_MODEL_NAME,
        llm_model_name: str = LLM_MODEL_NAME,
        top_k: int = TOP_K_CHUNKS,
    ):
        self.vectorstore_dir = vectorstore_dir
        self.embedding_model_name = embedding_model_name
        self.llm_model_name = llm_model_name
        self.top_k = top_k

        self.vectorstore = self._load_vectorstore()
        self.llm = self._load_llm()
        self.memory = ConversationBufferMemory(
            memory_key="chat_history", return_messages=True, output_key="answer"
        )
        self.chain = self._build_chain()

    # -----------------------------------------------------------------
    def _load_vectorstore(self):
        """
        Loads the FAISS vector store saved by ingest.py.
        Raises a clear error if the vector store was never created.
        """
        if not os.path.isdir(self.vectorstore_dir):
            raise FileNotFoundError(
                f"Vector store not found at '{self.vectorstore_dir}'. "
                f"Please run 'python ingest.py' first to build it."
            )

        print(f"[INFO] Loading embedding model '{self.embedding_model_name}'...")
        embeddings = HuggingFaceEmbeddings(model_name=self.embedding_model_name)

        print(f"[INFO] Loading FAISS vector store from '{self.vectorstore_dir}'...")
        try:
            vectorstore = FAISS.load_local(
                self.vectorstore_dir,
                embeddings,
                # Required by newer versions of langchain/FAISS for local pickle loading.
                # Safe here because we created this file ourselves in ingest.py.
                allow_dangerous_deserialization=True,
            )
        except Exception as e:
            raise RuntimeError(f"Failed to load FAISS vector store: {e}")

        return vectorstore

    # -----------------------------------------------------------------
    def _load_llm(self):
        """
        Loads a free HuggingFace sequence-to-sequence model
        (flan-t5-base by default) and wraps it as a LangChain LLM.
        """
        print(f"[INFO] Loading LLM '{self.llm_model_name}' (first run may take a while)...")
        try:
            tokenizer = AutoTokenizer.from_pretrained(self.llm_model_name)
            model = AutoModelForSeq2SeqLM.from_pretrained(self.llm_model_name)

            hf_pipeline = pipeline(
                "text2text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=256,
                temperature=0.3,
                do_sample=False,
            )
            llm = HuggingFacePipeline(pipeline=hf_pipeline)
        except Exception as e:
            raise RuntimeError(
                f"Failed to load LLM '{self.llm_model_name}'. "
                f"Check your internet connection or try a smaller model. Error: {e}"
            )
        return llm

    # -----------------------------------------------------------------
    def _build_chain(self):
        """
        Builds a ConversationalRetrievalChain, which automatically:
          - rewrites follow-up questions using chat history
          - retrieves relevant chunks from FAISS
          - feeds them plus chat history into the LLM via our custom prompt
        """
        retriever = self.vectorstore.as_retriever(search_kwargs={"k": self.top_k})

        qa_prompt = PromptTemplate(
            template=QA_PROMPT_TEMPLATE,
            input_variables=["context", "chat_history", "question"],
        )

        chain = ConversationalRetrievalChain.from_llm(
            llm=self.llm,
            retriever=retriever,
            memory=self.memory,
            combine_docs_chain_kwargs={"prompt": qa_prompt},
            return_source_documents=True,
        )
        return chain

    # -----------------------------------------------------------------
    def ask(self, question: str) -> dict:
        """
        Sends a question to the chatbot and returns a dictionary containing
        the answer and the source document chunks used to generate it.

        Handles empty questions gracefully.
        """
        if not question or not question.strip():
            return {
                "answer": "Please enter a question before sending.",
                "sources": [],
            }

        try:
            result = self.chain.invoke({"question": question})
        except Exception as e:
            return {
                "answer": f"Sorry, something went wrong while generating a response: {e}",
                "sources": [],
            }

        sources = []
        for doc in result.get("source_documents", []):
            source_name = doc.metadata.get("source", "unknown")
            sources.append(source_name)

        return {
            "answer": result.get("answer", "").strip(),
            "sources": list(dict.fromkeys(sources)),  # de-duplicate, keep order
        }

    def clear_memory(self):
        """Clears the conversation history, starting a fresh conversation."""
        self.memory.clear()


# ---------------------------------------------------------------------------
# Quick manual test when running this file directly:  python chatbot.py
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    bot = ChatBot()
    print("\nContext-Aware Chatbot ready! Type 'exit' to quit.\n")
    while True:
        user_input = input("You: ")
        if user_input.strip().lower() in ("exit", "quit"):
            break
        response = bot.ask(user_input)
        print(f"Bot: {response['answer']}")
        if response["sources"]:
            print(f"(Sources: {', '.join(response['sources'])})")


In [ ]:
%%writefile ContextAwareChatbot/app.py
"""
app.py
------
Streamlit front-end for the Context-Aware Chatbot.

Run with:
    streamlit run app.py

Features:
- Project title and description
- Sidebar with conversation history and a "Clear chat" button
- Text input box + Send button
- Nicely formatted chat bubbles for user and bot messages
- Shows which source documents were used to answer each question
- Handles missing vector store / empty questions gracefully
"""

import os
import streamlit as st

from chatbot import ChatBot

# ---------------------------------------------------------------------------
# PAGE CONFIGURATION
# ---------------------------------------------------------------------------
st.set_page_config(
    page_title="Context-Aware RAG Chatbot",
    page_icon="💬",
    layout="wide",
)


# ---------------------------------------------------------------------------
# CACHED CHATBOT LOADER
# ---------------------------------------------------------------------------
@st.cache_resource(show_spinner=True)
def load_chatbot():
    """
    Loads the ChatBot once and caches it across Streamlit re-runs,
    so the embedding model and LLM are not reloaded on every interaction.
    """
    return ChatBot()


# ---------------------------------------------------------------------------
# SESSION STATE INITIALISATION
# ---------------------------------------------------------------------------
if "chat_history" not in st.session_state:
    # Each item is a dict: {"role": "user"/"bot", "content": str, "sources": list}
    st.session_state.chat_history = []


# ---------------------------------------------------------------------------
# SIDEBAR
# ---------------------------------------------------------------------------
with st.sidebar:
    st.header("📚 About this project")
    st.markdown(
        """
        **Context-Aware Chatbot** built with:
        - LangChain
        - FAISS (vector database)
        - Sentence-Transformers embeddings
        - HuggingFace Transformers (flan-t5-base)
        - Streamlit

        Upload documents into the `documents/` folder and run
        `python ingest.py` before chatting.
        """
    )

    st.divider()
    st.subheader("🕑 Conversation History")
    if len(st.session_state.chat_history) == 0:
        st.caption("No messages yet.")
    else:
        for msg in st.session_state.chat_history:
            role_label = "🧑 You" if msg["role"] == "user" else "🤖 Bot"
            st.caption(f"**{role_label}:** {msg['content'][:60]}")

    st.divider()
    if st.button("🗑️ Clear chat", use_container_width=True):
        st.session_state.chat_history = []
        # Also clear the LangChain memory buffer, not just the UI history
        if "chatbot_instance" in st.session_state:
            st.session_state.chatbot_instance.clear_memory()
        st.rerun()


# ---------------------------------------------------------------------------
# MAIN AREA
# ---------------------------------------------------------------------------
st.title("💬 Context-Aware Chatbot (LangChain + RAG)")
st.markdown(
    "Ask questions about the documents in the `documents/` folder. "
    "This chatbot remembers previous questions, so you can ask natural "
    "follow-ups like *'Where was he born?'* after asking about a person."
)

# Try to load the chatbot; handle the case where ingest.py hasn't been run yet
try:
    if "chatbot_instance" not in st.session_state:
        with st.spinner("Loading models and vector store... this can take a minute the first time."):
            st.session_state.chatbot_instance = load_chatbot()
    bot = st.session_state.chatbot_instance
    load_error = None
except FileNotFoundError as e:
    bot = None
    load_error = str(e)
except Exception as e:
    bot = None
    load_error = f"Unexpected error while loading the chatbot: {e}"

if load_error:
    st.error(load_error)
    st.info("Run `python ingest.py` in your terminal or Colab cell, then restart this app.")
    st.stop()

# ---------------------------------------------------------------------------
# DISPLAY CHAT HISTORY
# ---------------------------------------------------------------------------
for msg in st.session_state.chat_history:
    if msg["role"] == "user":
        with st.chat_message("user"):
            st.write(msg["content"])
    else:
        with st.chat_message("assistant"):
            st.write(msg["content"])
            if msg.get("sources"):
                st.caption(f"📄 Source(s): {', '.join(msg['sources'])}")

# ---------------------------------------------------------------------------
# CHAT INPUT
# ---------------------------------------------------------------------------
user_question = st.chat_input("Type your question here and press Enter...")

if user_question is not None:
    # Handle empty / whitespace-only questions gracefully
    if not user_question.strip():
        st.warning("Please enter a question before sending.")
    else:
        # Show the user's message immediately
        st.session_state.chat_history.append({"role": "user", "content": user_question})
        with st.chat_message("user"):
            st.write(user_question)

        # Generate and show the bot's response
        with st.chat_message("assistant"):
            with st.spinner("Thinking..."):
                response = bot.ask(user_question)
            st.write(response["answer"])
            if response["sources"]:
                st.caption(f"📄 Source(s): {', '.join(response['sources'])}")

        st.session_state.chat_history.append(
            {
                "role": "bot",
                "content": response["answer"],
                "sources": response["sources"],
            }
        )


In [ ]:
%%writefile ContextAwareChatbot/README.md
# Context-Aware Chatbot Using LangChain and Retrieval-Augmented Generation (RAG)

## 📌 Project Overview

This project is a **context-aware chatbot** that answers questions using information
retrieved from a set of documents (Retrieval-Augmented Generation, or RAG), while also
remembering previous turns of the conversation.

It is built entirely with **free, open-source tools** so it can run on **Google Colab's
free tier** without any paid API keys (no OpenAI, no Pinecone, no paid vector database).

## ✨ Features

- **Retrieval-Augmented Generation (RAG):** answers are grounded in your own documents,
  not just the model's training data.
- **Conversational memory:** the bot remembers earlier messages, so follow-up questions
  like "Where was he born?" are understood correctly.
- **Fully free stack:** HuggingFace `flan-t5-base` LLM, `all-MiniLM-L6-v2` embeddings,
  and a local FAISS vector database.
- **Streamlit web UI:** chat interface with history, a clear-chat button, and source
  citations for every answer.
- **Sample documents included:** no dataset download required — 4 ready-to-use text
  files about AI, Machine Learning, Python, and LangChain are already in `documents/`.
- **Error handling:** missing documents, empty questions, a missing vector store, and
  model-loading failures are all handled with clear messages.

## 📁 Folder Structure

```
ContextAwareChatbot/
│
├── app.py                 # Streamlit web application
├── chatbot.py              # Core RAG + memory chatbot logic
├── ingest.py                # Builds the FAISS vector store from documents/
├── requirements.txt         # Python dependencies
├── README.md                 # This file
├── documents/                # Source documents used for retrieval
│   ├── ai.txt
│   ├── ml.txt
│   ├── python.txt
│   └── langchain.txt
└── vectorstore/               # Generated FAISS index (created by ingest.py)
```

## 📦 Where the "dataset" comes from

This project does not need you to find or download any external dataset. Four short,
self-contained text documents (on AI, Machine Learning, Python, and LangChain) are
generated directly inside `documents/` — either already included in this project, or
created automatically for you by **Cell 3** of the Colab notebook. This means the whole
project runs end-to-end with **no manual downloads and no internet dataset dependency**.

If you'd rather use your **own** documents (PDF or TXT), just drop them into the
`documents/` folder (in Colab: use the file upload button in the left sidebar, or
`files.upload()` in a cell) and re-run `ingest.py`. Any `.txt` or `.pdf` file works.

## 🛠️ Installation

1. Clone or download this project folder.
2. Install dependencies:

```bash
pip install -r requirements.txt
```

> On Google Colab, use `!pip install -r requirements.txt` in a code cell instead.

## ▶️ Running `ingest.py`

This builds the FAISS vector database from the documents in `documents/`. Run it once
(and again any time you add or change documents):

```bash
python ingest.py
```

Expected output:

```
[INFO] Loaded 4 document(s) from 'documents'.
[INFO] Split documents into NN chunks.
[INFO] Loading embedding model 'sentence-transformers/all-MiniLM-L6-v2'...
[INFO] Creating FAISS vector store from chunks...
[INFO] Vector store saved to 'vectorstore'.
[SUCCESS] Ingestion complete! You can now run chatbot.py or app.py.
```

## ▶️ Running `app.py`

Launch the Streamlit chat interface:

```bash
streamlit run app.py
```

On Google Colab, Streamlit needs a tunnel to be viewable in the browser — see the
provided Colab notebook for a ready-made command using `localtunnel`.

You can also test the chatbot directly from the terminal, without Streamlit:

```bash
python chatbot.py
```

## 💬 Example Conversation

```
You: What is machine learning?
Bot: Machine Learning is a subset of Artificial Intelligence that gives computers
     the ability to learn from data and improve their performance over time without
     being explicitly programmed for every task.
(Sources: ml.txt)

You: What are its main types?
Bot: The main types are supervised learning, unsupervised learning, and
     reinforcement learning.
(Sources: ml.txt)

You: Which one uses labeled data?
Bot: Supervised learning uses labeled data, where each training example has an
     input and a known correct output.
(Sources: ml.txt)
```

Notice how the third question ("Which one uses labeled data?") relies on the
conversation history to understand "one" refers to the types of ML mentioned earlier —
this is the conversational memory in action.

## 🚀 Future Improvements

- Swap `flan-t5-base` for a larger instruction-tuned model (e.g. `flan-t5-large` or a
  quantized Llama/Mistral model) when running on a GPU runtime for higher quality answers.
- Add support for `.docx` and website URLs as document sources.
- Add a re-ranking step to improve retrieval precision on larger document sets.
- Persist chat history to disk/database so conversations survive app restarts.
- Add streaming token-by-token responses in the Streamlit UI.
- Add automated evaluation of answer quality (e.g. using RAGAS).

## 🧠 Tech Stack Summary

| Component        | Tool                                             |
|-------------------|---------------------------------------------------|
| Orchestration      | LangChain                                         |
| LLM                 | google/flan-t5-base (HuggingFace, free)            |
| Embeddings          | sentence-transformers/all-MiniLM-L6-v2             |
| Vector Database      | FAISS (local, free)                              |
| Memory                | LangChain ConversationBufferMemory              |
| Document Loading         | PyPDF, LangChain TextLoader                  |
| UI                          | Streamlit                                |
| Environment                   | Google Colab (Free tier compatible) |


In [ ]:
print('Sample documents and full project files created.')
print(os.listdir(DOCS_DIR))


## 4. Generate embeddings

Load the free, local `sentence-transformers/all-MiniLM-L6-v2` embedding model and the source documents, then split them into chunks ready to be embedded.


In [ ]:
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

# Load documents
loader = DirectoryLoader(DOCS_DIR, glob='**/*.txt', loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
documents = loader.load()
print(f'Loaded {len(documents)} document(s).')

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f'Split into {len(chunks)} chunks.')

# Load the embedding model
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
print('Embedding model loaded.')


## 5. Build FAISS vector store


In [ ]:
vectorstore = FAISS.from_documents(chunks, embeddings)
print('FAISS vector store built with', vectorstore.index.ntotal, 'vectors.')


## 6. Test the chatbot (with conversational memory)

Load the free `google/flan-t5-base` model, wrap it as a LangChain LLM, and build a `ConversationalRetrievalChain` with `ConversationBufferMemory` so the bot remembers earlier turns.


In [ ]:
LLM_MODEL_NAME = 'google/flan-t5-base'

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL_NAME)

hf_pipeline = pipeline(
    'text2text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.3,
    do_sample=False,
)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

QA_PROMPT_TEMPLATE = '''You are a helpful assistant that answers questions using the
provided context from the user's documents. If the answer is not contained in the
context, say "I could not find that information in the provided documents." Do not
make up information.

Context:
{context}

Chat History:
{chat_history}

Question: {question}
Helpful Answer:'''

qa_prompt = PromptTemplate(template=QA_PROMPT_TEMPLATE, input_variables=['context', 'chat_history', 'question'])

memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True, output_key='answer')

chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={'k': 3}),
    memory=memory,
    combine_docs_chain_kwargs={'prompt': qa_prompt},
    return_source_documents=True,
)

print('Chatbot chain is ready!')


In [ ]:
# Test 1: ask about a topic
result1 = chain.invoke({'question': 'What is machine learning?'})
print('Q: What is machine learning?')
print('A:', result1['answer'])
print()

# Test 2: a follow-up question that relies on memory
result2 = chain.invoke({'question': 'What are its main types?'})
print('Q: What are its main types?')
print('A:', result2['answer'])


## 7. Save the vector database


In [ ]:
VECTORSTORE_DIR = os.path.join(PROJECT_DIR, 'vectorstore')
vectorstore.save_local(VECTORSTORE_DIR)
print(f'Vector store saved to {VECTORSTORE_DIR}')


## 8. Download project files

Zip up the entire `ContextAwareChatbot/` project (code, documents, and the saved vector store) and download it to your computer.


In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive('ContextAwareChatbot', 'zip', PROJECT_DIR)
print(f'Created archive: {zip_path}')
files.download(zip_path)


## (Optional) Run the Streamlit app from Colab

Streamlit apps need a tunnel to be viewed from Colab's browser environment. One free option is `localtunnel`:

```python
!npm install -g localtunnel -q
!streamlit run ContextAwareChatbot/app.py &>/content/logs.txt &
import time; time.sleep(5)
!npx localtunnel --port 8501
```

Then open the printed URL in your browser (the tunnel password is your Colab machine's public IP, printed by `!wget -q -O - ipv4.icanhazip.com`).
